In [1]:
# Build the local cache: egsl subset · 1990-09-01 → 2020-09-01 · daily · all variables in one netCDF
# Shape lineage (egsl = St. Lawrence Gulf and Estuary): ~/data/masks/climatology_bbox/egsl/README.md
import time
import datetime as dt
from pathlib import Path

import geopandas as gpd
import xarray as xr
from siphon.catalog import TDSCatalog
from clisops.core import subset

CATALOG_URL = "https://pavics.ouranos.ca/twitcher/ows/proxy/thredds/catalog/datasets/reanalyses/catalog.xml"
DATASET_ID = "day_NAM_GovCan_CaSR_v32_1968-2024.ncml"
SHAPE_PATH = "/notebook_dir/writable-workspace/egsl_32198_bbox.geojson"
CLIM_START, CLIM_END = "1990-09-01", "2020-09-01"
VARIABLES = ["tas", "tasmax", "tasmin", "pr", "cfia", "uas", "vas"]  # uas/vas: vector wind averaging
OUT_PATH = Path("/notebook_dir/writable-workspace/casr_v32_day_egsl_19900901-20200901.nc")
COMPRESSION = {"zlib": True, "complevel": 4}

opendap_url = TDSCatalog(CATALOG_URL).datasets[DATASET_ID].access_urls["OPENDAP"]
ds = xr.open_dataset(opendap_url, chunks="auto")[VARIABLES]
egsl = subset.subset_shape(ds, shape=gpd.read_file(SHAPE_PATH))
egsl = egsl.sel(time=slice(CLIM_START, CLIM_END))   # applies only to time-dependent variables; sftof is static

egsl.attrs |= {
    "cache_source": opendap_url,
    "cache_window": f"{CLIM_START}/{CLIM_END}",
    "cache_shape": SHAPE_PATH,
    "cache_created": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
}
for var in egsl.data_vars:
    egsl[var].encoding = {}                          # drop OPeNDAP-side encoding so COMPRESSION applies cleanly

tmp = OUT_PATH.with_suffix(".nc.part")              # an interrupted write never looks complete
t0 = time.perf_counter()
egsl.to_netcdf(tmp, encoding={var: COMPRESSION for var in egsl.data_vars})
write_s = time.perf_counter() - t0
tmp.rename(OUT_PATH)

for var in egsl.data_vars:
    print(f"{var:<8} {egsl[var].dims}  {egsl[var].nbytes/1e9:5.2f} GB")
print(f"total {egsl.nbytes/1e9:.2f} GB in {write_s:.0f} s ({egsl.nbytes/1e6/write_s:.1f} MB/s) "
      f"→ {OUT_PATH.stat().st_size/1e9:.2f} GB on disk")


tas      ('time', 'rlat', 'rlon')   0.70 GB
tasmax   ('time', 'rlat', 'rlon')   0.70 GB
tasmin   ('time', 'rlat', 'rlon')   0.70 GB
pr       ('time', 'rlat', 'rlon')   0.70 GB
cfia     ('time', 'rlat', 'rlon')   0.70 GB
uas      ('time', 'rlat', 'rlon')   0.70 GB
vas      ('time', 'rlat', 'rlon')   0.70 GB
total 4.87 GB in 667 s (7.3 MB/s) → 1.33 GB on disk
